# Univariate Linear Regression (Scratch vs scikit-learn)
We predict a **proxy** target: `days_until_failure`.

- Feature: one axis current (e.g., `Axis #1`)
- Target: time remaining until the last timestamp in the dataset

This matches the workshop requirement of univariate regression and sets up Session 2 (alert within 14 days).

In [ ]:
import numpy as np

from src.data_loader import load_from_csv, make_multi_robot
from src.preprocessing import (
    clean_robot_current_df,
    add_days_until_failure_target,
    select_feature_target,
    train_test_split,
    standardize_univariate,
)
from src.model import ScratchLinearRegression, fit_sklearn_linear_regression
from src.evaluation import rmse, mae, r2_score

# Load + (optional) create 3 robot sources
df = load_from_csv('data/raw/RMBR4-2_export_test.csv')
df = make_multi_robot(df)

# Clean + create regression target
df = clean_robot_current_df(df)
df = add_days_until_failure_target(df, target_col='days_until_failure')

# Choose one feature (univariate)
feature_col = 'Axis #1'
X, y = select_feature_target(df, feature_col, 'days_until_failure')

# Split + standardize
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, seed=42)
X_train, X_test, mu, sigma = standardize_univariate(X_train, X_test)

print('Train size:', len(X_train), 'Test size:', len(X_test))
print('Feature standardized with mu=', mu, 'sigma=', sigma)


In [ ]:
# Scratch model
scratch = ScratchLinearRegression(learning_rate=0.01, n_iters=2000).fit(X_train, y_train)
y_pred_scratch = scratch.predict(X_test)

print('Scratch RMSE:', rmse(y_test, y_pred_scratch))
print('Scratch MAE :', mae(y_test, y_pred_scratch))
print('Scratch R2  :', r2_score(y_test, y_pred_scratch))

In [ ]:
# scikit-learn model
sk = fit_sklearn_linear_regression(X_train, y_train)
y_pred_sk = sk.predict(X_test)

print('Sklearn RMSE:', rmse(y_test, y_pred_sk))
print('Sklearn MAE :', mae(y_test, y_pred_sk))
print('Sklearn R2  :', r2_score(y_test, y_pred_sk))

In [ ]:
# 2-weeks alert demo (Session 2 idea)
alerts = int(np.sum(y_pred_sk <= 14.0))
print(f'Alerts (predicted <= 14 days): {alerts} / {len(y_pred_sk)}')